# 第 39 课：置信度、N-best 与语义重排序

系统不仅要给答案，还要知道何时不确定，并保留足够候选让语义模块纠错。

<!-- course-upgrade-v2 -->
## 学习导航

| 项目 | 内容 |
|---|---|
| 所属阶段 | 后处理与语义 |
| 建议投入 | 3～5 小时，可分 2～3 次完成 |
| 前置要求 | 完成第 38 课；如果前测低于 2/3，先回看上一课小结 |
| 本课核心 | 置信度校准、N-best、语义重排 |
| 完成标准 | 能口头解释核心概念；独立完成强化题；从空白重写核心函数 |

高效顺序：**先回答前测 → 预测代码结果 → 再运行 → 修改一个变量 → 关闭答案复现 → 次日回忆。**


<!-- course-upgrade-v2 -->
## 课前诊断（先不要运行代码）

1. 分别用一句话解释：置信度校准、N-best、语义重排。
2. 画出这三个概念之间的输入—输出关系。
3. 写下你最不确定的一点，并给出一个暂时猜测。

自评：答对 0～1 题先复习前置课；答对 2 题可以正常学习；3 题都能讲清楚则直接挑战代码和迁移题。


<!-- course-bridge-v3 -->
## 知识接力：先取回旧知识，再进入本课

### 3 分钟闭卷回忆

在新 Markdown cell 中回答，**不要先翻前文**：raw ASR 与 normalized text 的证据边界；置信度；N-best；会话状态隔离。

- 三项都能用“含义 + 单位/shape + 一个数字例子”回答：进入本课。
- 能回答两项：学习本课，但把缺口记入 `LEARNING_LOG.md`。
- 只能回答零到一项：先回到 [上一课](38_标点恢复_ITN与文本规范化.ipynb)与[唯一学习路径](../LEARNING_PATH.md)，做一次最小实验；不要靠继续看新术语掩盖断点。

### 本课接口契约

```text
输入：带时间、说话人、候选与置信度的可追溯识别结果
  ↓ 本课要学会的变换、状态或判断
输出：保留原证据、可校准、可拒绝或澄清的文本/语义结果
```

学完后必须能解释：输入的哪个单位/shape/状态若丢失，会让输出“仍能运行却语义错误”。


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import soundfile as sf
import librosa

def find_root():
    here=Path.cwd().resolve()
    for p in [here,*here.parents]:
        if (p/"pyproject.toml").exists():return p
    raise FileNotFoundError("请从 learn_asr 或 notebooks 目录启动 Jupyter")
ROOT=find_root();plt.rcParams["figure.figsize"]=(11,4)
print("项目根目录:",ROOT)

from scipy.special import softmax
from ipywidgets import interact,FloatSlider

## 1. 最大 softmax 不是天然校准的置信度

In [ ]:
logits=np.array([[4,1,0],[2.2,2.0,0],[10,8,0]],float)
for z in logits:
    p=softmax(z);entropy=-np.sum(p*np.log(p+1e-12));print("p",p,"max",p.max(),"entropy",entropy)

模型可能过度自信。需要在独立数据上做 reliability diagram、ECE，并可使用 temperature scaling；句级置信度还要处理长度、blank 和 beam/lattice posterior。

## 2. N-best 语义重排序

In [ ]:
cands=[{"text":"打开空调","asr":-2.2,"intent":.92},{"text":"打开空道","asr":-1.9,"intent":.08},{"text":"打卡空调","asr":-2.0,"intent":.12}]
@interact(weight=FloatSlider(min=0,max=3,value=1,step=.1,description="semantic weight"))
def rerank(weight=1):
    for c in cands:c["score"]=c["asr"]+weight*np.log(c["intent"]+1e-6)
    for c in sorted(cands,key=lambda x:x["score"],reverse=True):print(c)

语义重排只能在声学/搜索仍保留正确候选时起作用。若正确文本已被 beam 剪掉，后端无法凭空恢复而不承担幻觉风险。

## 3. Reject/clarify 是合法输出

低置信度时可以请求复述、展示候选、转人工或只执行可撤销操作。高风险命令不能因为“语义看起来合理”就忽略声学不确定性。

## 本课测试

1. max softmax=0.99 是否保证 99% 正确？
2. N-best 为什么比 1-best 更适合后处理？
3. semantic weight 太大会怎样？
4. 正确候选被 beam 删除后还能可靠重排回来吗？
5. 低置信度时系统必须强行给一个答案吗？

<details><summary>展开参考答案</summary>

1. 不保证，需要校准。2. 保留替代假设。3. 可能无视声音选择语义常见句。4. 不能。5. 不必，可以拒识或澄清。

</details>

<!-- course-upgrade-v2 -->
## 强化练习：第 39 课专属题库

请先把答案写进新的 Markdown/Code cell，再展开自评标准。

### A. 基础回忆

1. 不看上文，分别定义 `置信度校准`、`N-best`、`语义重排`。
2. 哪一个量/状态是本课最容易在模块边界丢失的？它的单位和 shape 是什么？
3. 本课至少写出两个“看起来能运行，但结果其实错误”的例子。

### B. 预测与推理

4. 场景：**NLU 高置信掩盖低声学置信**。先预测现象，再说明原因，最后给出一项可以验证猜测的指标。
5. 改变本课最关键参数的 0.5×、1×、2×，分别预测准确率、延迟、内存或数值误差怎样变化。
6. 画一张最小数据流图，在每条边标出 dtype、shape、时间单位或概率/代价方向。

### C. 编程与排错

7. 编程任务：**画 reliability diagram 并计算 ECE**。至少加入正常、边界、错误输入三类测试。
8. 故意制造一个 off-by-one、shape、状态未 reset 或数值稳定性错误；记录错误现象和定位过程。
9. 不看本课实现，从空白 cell 重写最核心函数，并用原实现作数值对照。

### D. 迁移与表达

10. 跨课任务：**连接 lattice 候选与拒识/澄清策略**。
11. 用 90 秒向没有学过 ASR 的人解释本课；禁止只念术语，必须举一个数字或生活例子。
12. 写出一个生产系统中会监控的指标，以及它异常时优先检查的三处位置。

<details><summary>展开自评标准</summary>

- 每题 0～2 分：0=无法回答；1=方向正确但缺少单位、边界或验证；2=解释完整且能用代码/数字验证。
- 24 分满分：达到 19 分再进入下一课；15～18 分次日重做错题；低于 15 分回看本课图和核心代码。
- 第 4 题必须包含“预测—原因—指标”，第 7～9 题必须真正运行测试，第 10 题必须明确上下游 contract。
- 核心答案至少应正确使用：置信度校准、N-best、语义重排。

</details>


<!-- course-upgrade-v2 -->
## 间隔复习与离场票

### 离场票（现在完成）

- [ ] 我能不用笔记解释 置信度校准、N-best、语义重排。
- [ ] 我能说出本课最常见的错误及其观测现象。
- [ ] 我能从空白重写一个核心函数，并通过至少 3 个测试。
- [ ] 我能说明本课对上一层和下一层接口的影响。

### 复习时间表

- **明天（5 分钟）**：闭卷写出三个核心概念和一个公式/shape。
- **7 天后（15 分钟）**：重做第 4、7、10 题，不运行原答案。
- **30 天后（20 分钟）**：从真实音频或随机张量重新构造一个最小实验。

把错题记录到根目录 `LEARNING_LOG.md`。不要只写“不会”，要写：原判断、证据、正确规则、下次检查动作。
